In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ============================================================
# STEP 1: Create the original dataset
# ============================================================
# 6 samples, 2 original features.
# Each row represents one sample.
# Each column represents one feature.

X = np.array([
    [ 99, -1],
    [ 98, -1],
    [ 97, -2],
    [101,  1],
    [102,  1],
    [103,  2]
], dtype=float)

print("Original data:")
print(X)
print("Shape:", X.shape)  # (6 samples, 2 features)


# ============================================================
# STEP 2: Center the data
# ============================================================
# PCA subtracts the mean of each feature.
# This moves the center of the data to (0, 0).

mean = np.mean(X, axis=0)
X_centered = X - mean

print("\nFeature means:")
print(mean)

print("\nCentered data:")
print(X_centered)


# ============================================================
# STEP 3: Calculate the covariance matrix
# ============================================================
# The covariance matrix tells us how features vary
# individually and how they vary together.
#
# Rows = samples, columns = features.
# rowvar=False tells NumPy that columns are features.

cov_matrix = np.cov(X_centered, rowvar=False)

print("\nCovariance matrix:")
print(cov_matrix)


# ============================================================
# STEP 4: Find eigenvalues and eigenvectors
# ============================================================
# This is where PCA finds the principal directions.
#
# Each eigenvector contains weights for a new feature.
# Its corresponding eigenvalue tells us how much variance
# lies along that direction.
#
# eigh() is suitable for a symmetric covariance matrix.
# It returns eigenvalues in ascending order.

eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort eigenvalues from largest to smallest.
# Apply the same ordering to their eigenvectors.
order = np.argsort(eigenvalues)[::-1]

eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]

print("\nEigenvalues (largest first):")
print(eigenvalues)

print("\nEigenvectors (each column is a direction):")
print(eigenvectors)

# The first column is the direction of PC1.
# The second column is the direction of PC2.
#
# These weights were calculated from the data,
# not selected manually.

pc1 = eigenvectors[:, 0]
pc2 = eigenvectors[:, 1]

print("\nPC1 weights:")
print(pc1)

print("\nPC2 weights:")
print(pc2)


# ============================================================
# STEP 5: Calculate explained variance
# ============================================================
# The larger the eigenvalue, the more variance that component
# captures.
#
# Explained variance ratio = component eigenvalue /
#                            sum of all eigenvalues.

explained_variance_ratio = (
    eigenvalues / np.sum(eigenvalues)
)

print("\nExplained variance ratio:")
print(explained_variance_ratio)

print("PC1 captures:",
      round(explained_variance_ratio[0] * 100, 2), "%")

print("PC2 captures:",
      round(explained_variance_ratio[1] * 100, 2), "%")


# ============================================================
# STEP 6: Create the new features
# ============================================================
# Put the two principal directions into a matrix.
# Each column represents one principal direction.
#
# Shape of eigenvectors: (2 original features, 2 PCs)
#
# New data = centered data @ principal directions

W = eigenvectors

Z = X_centered @ W

print("\nPrincipal direction matrix W:")
print(W)

print("\nNew features Z:")
print(Z)

print("Shape of Z:", Z.shape)  # (6, 2)
#
# Z[:, 0] contains the new feature z1 for all samples.
# Z[:, 1] contains the new feature z2 for all samples.
#
# For each sample:
# z1 = x1_centered * pc1_weight_1
#    + x2_centered * pc1_weight_2
#
# The same calculation creates z2 using PC2 weights.


# ============================================================
# STEP 7: Reduce from 2 features to 1 feature
# ============================================================
# Keep only the first principal component.
# We discard the second component because it captures
# very little variance in this dataset.

W_1 = eigenvectors[:, :1]

Z_1 = X_centered @ W_1

print("\nData reduced to one feature:")
print(Z_1)

print("Shape:", Z_1.shape)  # (6, 1)
#
# Every sample now has one new value instead of two.
# This value is its coordinate along the PC1 direction.


# ============================================================
# STEP 8: Reconstruct the original data approximately
# ============================================================
# Convert the one-dimensional representation back into
# the original two-feature space.
#
# This reconstruction is approximate because PC2 was removed.

X_reconstructed = Z_1 @ W_1.T + mean

print("\nReconstructed data using PC1 only:")
print(X_reconstructed)


# ============================================================
# STEP 9: Compare with scikit-learn PCA
# ============================================================
# sklearn performs centering automatically.
# It does NOT automatically standardize every feature
# to unit variance.

pca = PCA(n_components=2)

Z_sklearn = pca.fit_transform(X)

print("\nScikit-learn component weights:")
print(pca.components_)

print("\nScikit-learn explained variance ratios:")
print(pca.explained_variance_ratio_)

print("\nScikit-learn transformed data:")
print(Z_sklearn)

# Eigenvector signs may be reversed between implementations.
# A direction u and its opposite -u describe the same axis,
# so the signs of the resulting coordinates can differ.


# ============================================================
# STEP 10: Visualize the original data and PC1
# ============================================================

plt.figure(figsize=(8, 6))

# Plot the original samples.
plt.scatter(
    X_centered[:, 0],
    X_centered[:, 1],
    label="Original samples"
)

# Draw the PC1 direction through the origin.
# The eigenvector has unit length, so multiply it by a
# convenient distance to make the direction visible.
plt.arrow(
    0, 0,
    pc1[0] * 3,
    pc1[1] * 3,
    head_width=0.15,
    length_includes_head=True,
    color="red",
    label="PC1 direction"
)

plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)

plt.xlabel("Centered feature 1")
plt.ylabel("Centered feature 2")
plt.title("PCA: Finding the First Principal Component")
plt.legend()
plt.grid(True)
plt.show()